# POD-ANN Stator Tooth EM-Force Surrogate
## `run_pod_prediction` — Interactive Notebook

**Five-step pipeline:**

| Step | What happens |
|------|-------------|
| 1 · Config | Set paths, target operating point, hyper-parameters |
| 2 · Initialise | Create `PODProcessor`; train or load a saved model |
| 3 · POD + Train | Decompose all FEM snapshots → train ANN (rpm, torque → T_k) |
| 4 · Predict | Reconstruct full force matrix for any operating condition |
| 5 · Validate | Compare against a FEM reference file *(optional)* |

> **Run all cells top-to-bottom.** Only **Step 1** needs editing.


---
## Step 0 · Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from class_file_improved import PODProcessor   # <- must be in the same folder

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True,
                     "grid.alpha": 0.35, "font.size": 11})
print("Imports OK ✓")


---
## Step 1 · Configuration
> ✏️ **Edit this cell only** — everything else runs automatically.


In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR   = Path("./fem_data")          # folder containing FEM xlsx files
MODEL_PATH = Path("./pod_ann_model.pt")  # where the trained model is saved

# ── Target operating point (interpolation allowed – no xlsx needed) ──────────
PREDICT_RPM    = 3500    # rpm
PREDICT_TORQUE = 75.0    # Nm

# ── Validation: set to an xlsx path to compare prediction vs FEM ─────────────
VALIDATION_XLSX  = None                  # e.g. Path("./fem_data/3500rpm_75Nm.xlsx")
VALIDATION_TOOTH = 0                     # 0-based tooth index to plot

# ── POD / ANN hyper-parameters ───────────────────────────────────────────────
MODE_COUNT = 5      # POD modes retained per operating point
EPOCHS     = 800    # ANN training epochs
LR         = 1e-3   # initial Adam learning rate

# ── Set True to reload a previously saved model and skip training ─────────────
LOAD_EXISTING_MODEL = False

# ── Helper ───────────────────────────────────────────────────────────────────
def print_metrics(label, metrics):
    print(f"\n{'─'*52}\n  {label}\n{'─'*52}")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k:<22s}: {v:.6f}")
        elif isinstance(v, list):
            print(f"  {k:<22s}: [{v[0]:.6f} … {v[-1]:.6f}]  (len={len(v)})")
        else:
            print(f"  {k:<22s}: {v}")

print("Configuration loaded ✓")
print(f"  DATA_DIR      : {DATA_DIR}")
print(f"  MODEL_PATH    : {MODEL_PATH}")
print(f"  TARGET OP     : {PREDICT_RPM} rpm  |  {PREDICT_TORQUE} Nm")
print(f"  MODE_COUNT    : {MODE_COUNT}   EPOCHS : {EPOCHS}   LR : {LR}")
print(f"  LOAD_EXISTING : {LOAD_EXISTING_MODEL}")


---
## Step 2 · Initialise Processor


In [ ]:
print("=" * 60)
print("  POD-ANN Stator Force Surrogate")
print("=" * 60)

proc = PODProcessor(mode_count=MODE_COUNT, epochs=EPOCHS, lr=LR)
print(f"\nPODProcessor created  (device: {proc.device})")


---
## Step 3 · Run POD Analysis + Train ANN
*(Skipped if `LOAD_EXISTING_MODEL = True` and the `.pt` file exists)*


In [ ]:
if LOAD_EXISTING_MODEL and MODEL_PATH.exists():
    # ── Load path ──────────────────────────────────────────────────────────
    print(f"[Load]  Loading model from '{MODEL_PATH}' …")
    proc.load(MODEL_PATH)
    print("[Load]  Done — skip to Step 4.")

else:
    # ── 3a: POD on every xlsx ──────────────────────────────────────────────
    print(f"[POD]  Scanning '{DATA_DIR}' for xlsx files …\n")
    pod_results = proc.run(DATA_DIR)

    print(f"\n[POD]  {len(pod_results)} operating point(s) collected:\n")
    print(f"  {'RPM':>6}  {'Torque (Nm)':>12}  {'Energy captured':>17}")
    print(f"  {'─'*6}  {'─'*12}  {'─'*17}")
    for (rpm, torque), res in pod_results.items():
        cum_e = (res['s'][:MODE_COUNT]**2).sum() / (res['s']**2).sum() * 100
        flag  = "  ⚠️  < 95 %" if cum_e < 95 else ""
        print(f"  {rpm:>6d}  {torque:>12.1f}  {cum_e:>14.2f} %{flag}")


In [ ]:
# ── 3b: Train ANN ─────────────────────────────────────────────────────────
if not (LOAD_EXISTING_MODEL and MODEL_PATH.exists()):
    print(f"\n[Train]  Training ANN for {EPOCHS} epochs …")
    history = proc.train_mode_predictor(test_size=0.2, random_state=42)

    print_metrics("ANN Test-Set Metrics",
                  {k: v for k, v in history.items() if not isinstance(v, list)})


In [ ]:
# ── 3c: Training loss curves ───────────────────────────────────────────────
if not (LOAD_EXISTING_MODEL and MODEL_PATH.exists()):
    epochs_ax = range(1, len(history["train_loss_history"]) + 1)

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.semilogy(epochs_ax, history["train_loss_history"],
                label="Train loss", linewidth=1.8)
    ax.semilogy(epochs_ax, history["val_loss_history"],
                label="Validation loss", linewidth=1.8, linestyle="--")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss (log scale)")
    ax.set_title("ANN Training History")
    ax.legend()
    plt.tight_layout()
    plt.savefig("training_history.png", dpi=150)
    plt.show()
    print("[Plot]  Training history saved to 'training_history.png'")


In [ ]:
# ── 3d: Save model ────────────────────────────────────────────────────────
if not (LOAD_EXISTING_MODEL and MODEL_PATH.exists()):
    proc.save(MODEL_PATH)
    print(f"[Save]  Model saved to '{MODEL_PATH}'")


---
## Step 4 · Predict Force Field for Target Operating Condition


In [ ]:
print(f"[Predict]  {PREDICT_RPM} rpm  |  {PREDICT_TORQUE} Nm")

U_pred, T_pred = proc.predict_modes(PREDICT_RPM, PREDICT_TORQUE)
A_pred         = proc.reconstruct_field(U_pred, T_pred)

n_teeth     = A_pred.shape[0] // 2
angle_steps = np.arange(A_pred.shape[1])

print(f"\nReconstructed matrix A_pred:")
print(f"  Shape : {A_pred.shape}")
print(f"  Rows  : {A_pred.shape[0]} DOF  =  {n_teeth} teeth × 2 (FR + FT)")
print(f"  Cols  : {A_pred.shape[1]} rotor-angle steps")


In [ ]:
# ── All-teeth overview plot ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

cmap = plt.cm.tab10
for tooth in range(n_teeth):
    c = cmap(tooth % 10)
    axes[0].plot(angle_steps, A_pred[tooth, :],
                 label=f"Tooth {tooth}", color=c, alpha=0.85)
    axes[1].plot(angle_steps, A_pred[n_teeth + tooth, :],
                 label=f"Tooth {tooth}", color=c, alpha=0.85)

axes[0].set_ylabel("Radial Force FR (normalised)")
axes[0].set_title(f"Predicted Radial Forces  |  {PREDICT_RPM} rpm  |  {PREDICT_TORQUE} Nm")
axes[0].legend(loc="upper right", fontsize=8, ncol=min(n_teeth, 4))

axes[1].set_ylabel("Tangential Force FT (normalised)")
axes[1].set_title(f"Predicted Tangential Forces  |  {PREDICT_RPM} rpm  |  {PREDICT_TORQUE} Nm")
axes[1].set_xlabel("Rotor angle step")
axes[1].legend(loc="upper right", fontsize=8, ncol=min(n_teeth, 4))

plt.suptitle("POD-ANN Prediction — All Teeth", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("predicted_forces_all_teeth.png", dpi=150)
plt.show()
print("[Plot]  Saved to 'predicted_forces_all_teeth.png'")


In [ ]:
# ── Spatial mode heatmap (U_mean) ─────────────────────────────────────────
fig, axes = plt.subplots(1, proc.mode_count, figsize=(14, 4), sharey=True)

for i, ax in enumerate(axes):
    col = U_pred[:, i]
    vmax = np.abs(col).max()
    ax.barh(range(len(col)), col,
            color=["steelblue" if v >= 0 else "tomato" for v in col])
    ax.set_title(f"Mode {i+1}", fontsize=10)
    ax.set_xlabel("Amplitude")
    ax.axvline(0, color="k", linewidth=0.8)
    if i == 0:
        ax.set_ylabel("DOF index  (FR teeth | FT teeth)")

plt.suptitle(f"Spatial Modes U_mean  ({proc.mode_count} modes)", fontweight="bold")
plt.tight_layout()
plt.savefig("spatial_modes.png", dpi=150)
plt.show()
print("[Plot]  Saved to 'spatial_modes.png'")


In [ ]:
# ── Temporal coefficients T_k ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))

for i in range(T_pred.shape[0]):
    ax.plot(angle_steps, T_pred[i, :], label=f"Mode {i+1}", linewidth=1.6)

ax.set_xlabel("Rotor angle step")
ax.set_ylabel("Temporal coefficient")
ax.set_title(f"Temporal Coefficients T_k  |  {PREDICT_RPM} rpm  |  {PREDICT_TORQUE} Nm")
ax.legend(ncol=proc.mode_count)
plt.tight_layout()
plt.savefig("temporal_coefficients.png", dpi=150)
plt.show()
print("[Plot]  Saved to 'temporal_coefficients.png'")


---
## Step 5 · Validate Against FEM Reference *(optional)*

Set `VALIDATION_XLSX` in **Step 1** to a known FEM file to enable this section.  
Leave it as `None` to skip — a summary message will be printed instead.


In [ ]:
if VALIDATION_XLSX is not None:
    print(f"[Validate]  Loading FEM reference: '{VALIDATION_XLSX}' …")

    df_val   = proc.read_forces(VALIDATION_XLSX)
    A_actual = df_val.to_numpy(dtype=float).T     # (n_dof, n_steps)

    tooth_metrics = proc.evaluate_single_tooth(
        A_actual    = A_actual,
        A_pred      = A_pred,
        tooth_index = VALIDATION_TOOTH,
        rpm         = PREDICT_RPM,
        torque      = PREDICT_TORQUE,
        n_teeth     = n_teeth,
    )
    print_metrics(f"Tooth {VALIDATION_TOOTH} — Validation Metrics", tooth_metrics)

else:
    print("[Validate]  VALIDATION_XLSX is None — FEM comparison skipped.")
    print("            Set VALIDATION_XLSX = Path('...') in Step 1 to enable.")


In [ ]:
# ── Per-tooth RMS table ────────────────────────────────────────────────────
if VALIDATION_XLSX is not None:
    fr_rms_all, ft_rms_all = [], []

    for t in range(n_teeth):
        fr_rms_all.append(float(np.sqrt(((A_actual[t, :]           - A_pred[t, :]           )**2).mean())))
        ft_rms_all.append(float(np.sqrt(((A_actual[n_teeth + t, :] - A_pred[n_teeth + t, :])**2).mean())))

    print(f"\nPer-tooth RMS error — all {n_teeth} teeth\n")
    print(f"  {'Tooth':>6}  {'FR RMS':>12}  {'FT RMS':>12}")
    print(f"  {'─'*6}  {'─'*12}  {'─'*12}")
    for t in range(n_teeth):
        print(f"  {t:>6d}  {fr_rms_all[t]:>12.4f}  {ft_rms_all[t]:>12.4f}")

    # ── Bar chart of per-tooth RMS ────────────────────────────────────────
    x = np.arange(n_teeth)
    width = 0.38

    fig, ax = plt.subplots(figsize=(max(8, n_teeth * 0.9), 4))
    ax.bar(x - width/2, fr_rms_all, width, label="FR RMS", color="steelblue")
    ax.bar(x + width/2, ft_rms_all, width, label="FT RMS", color="darkorange")
    ax.set_xticks(x)
    ax.set_xticklabels([f"T{i}" for i in range(n_teeth)])
    ax.set_xlabel("Tooth index")
    ax.set_ylabel("RMS error (normalised)")
    ax.set_title(f"Per-tooth Prediction Error  |  {PREDICT_RPM} rpm  |  {PREDICT_TORQUE} Nm")
    ax.legend()
    plt.tight_layout()
    plt.savefig("per_tooth_rms.png", dpi=150)
    plt.show()
    print("[Plot]  Saved to 'per_tooth_rms.png'")


---
## Done ✅

| Output file | Contents |
|---|---|
| `pod_ann_model.pt` | Trained model + scalers + U_mean |
| `training_history.png` | Train vs validation loss curves |
| `predicted_forces_all_teeth.png` | FR & FT waveforms — all teeth |
| `spatial_modes.png` | U_mean mode shapes |
| `temporal_coefficients.png` | T_k coefficient waveforms |
| `per_tooth_rms.png` | Per-tooth error bar chart *(validation only)* |
